# 03 — News Ticker Mapping and Export

## Purpose
Take the article-level sentiment dataset (one row per article) and produce the ticker-level news tables, keeping **only the 10 project tickers**.

Example: an article tagged `[AAPL, COST, BAC, BABA, AMZN]` becomes two rows, `AAPL` and `AMZN`. `COST`, `BAC` and `BABA` are discarded.

## Input
`data/processed/news_articles_sentiment.csv` (from `02_news_cleaning_and_finbert`). Its `symbols` column holds the **full** Alpaca ticker list, which is what this notebook filters.

## Outputs
| File | Grain | Contents |
|---|---|---|
| `news_tickers.csv` | one row per `(article_id, ticker)` | article ID and ticker only (the warehouse association table) |
| `news_clean.csv` | one row per `(article_id, ticker)` | the pair plus article metadata and FinBERT sentiment (flattened export for analysis / Power BI) |

## Important interpretation
FinBERT is **not** re-run here. The article's sentiment is copied to every ticker it mentions, so it describes the article text, not sentiment toward each specific company. Two columns help with this:
- `n_project_tickers`: how many of the 10 project tickers the article mentions.
- `n_symbols_total`: how many tickers Alpaca tagged in total. Articles tagged with dozens of tickers are usually market roundups, so consider down-weighting or excluding them in the analysis.

In [ ]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Paths (same ../../data convention as notebook 02) ----
PROCESSED_DIR = Path("../../data/processed")
INPUT_PATH = PROCESSED_DIR / "news_articles_sentiment.csv"
TICKERS_PATH = PROCESSED_DIR / "news_tickers.csv"
FLAT_PATH = PROCESSED_DIR / "news_clean.csv"

# ---- Canonical project tickers ----
TICKERS = ["AAPL", "JPM", "MSFT", "NVDA", "JNJ", "XOM", "KO", "AMZN", "LLY", "TSLA"]
PROJECT_TICKERS = set(TICKERS)
print("Project tickers:", sorted(PROJECT_TICKERS))

## 1. Load only the columns needed

`sentiment_text` is the largest column and is not needed here, so it is skipped. IDs are read as text so they are never converted to numbers.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Could not find {INPUT_PATH.resolve()}. Run notebook 02 first.")

WANTED = [
    "article_id", "published_at", "title", "summary", "source", "url", "symbols",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
]
articles = pd.read_csv(
    INPUT_PATH,
    usecols=lambda c: c in WANTED,
    dtype={"article_id": str, "symbols": str},
)
print(f"Loaded {len(articles):,} article-level rows")

required = {"article_id", "published_at", "title", "symbols", "finbert_label", "finbert_score"}
missing = required - set(articles.columns)
if missing:
    raise ValueError(f"Input is missing required columns: {sorted(missing)}")

# Optional columns: create empty ones so the export has a stable schema.
for col in WANTED:
    if col not in articles.columns:
        articles[col] = pd.NA

articles["article_id"] = articles["article_id"].str.strip()
if articles["article_id"].duplicated().any():
    raise ValueError("Input must have one row per unique article_id.")

articles["published_at"] = pd.to_datetime(articles["published_at"], utc=True, errors="coerce")
articles["finbert_score"] = pd.to_numeric(articles["finbert_score"], errors="coerce")

## 2. Parse the symbols and keep only project tickers

The `symbols` column stores the full ticker list as JSON (written by notebook 02). The parser also accepts Python-list-like strings and plain comma-separated text. Tickers are upper-cased and de-duplicated, then filtered to `PROJECT_TICKERS`.

In [ ]:
def parse_symbols(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        items = list(value)
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed = value.split(",")
        items = list(parsed) if isinstance(parsed, (list, tuple, set)) else [parsed]
    else:
        return []
    return list(dict.fromkeys(
        str(item).strip().upper() for item in items
        if item is not None and str(item).strip()
    ))


def keep_project_tickers(all_symbols):
    return sorted(t for t in all_symbols if t in PROJECT_TICKERS)


# Sanity check of the filtering rule, using the example from the project notes.
assert keep_project_tickers(parse_symbols('["AAPL", "COST", "BAC", "BABA", "AMZN"]')) == ["AAPL", "AMZN"]
assert keep_project_tickers(parse_symbols("tsla, nvda")) == ["NVDA", "TSLA"]
assert keep_project_tickers(parse_symbols("[]")) == []

all_symbols = articles["symbols"].apply(parse_symbols)
articles["n_symbols_total"] = all_symbols.apply(len)
articles["project_tickers"] = all_symbols.apply(keep_project_tickers)
articles["n_project_tickers"] = articles["project_tickers"].apply(len)
del all_symbols

has_project = articles["n_project_tickers"].gt(0)
print("Articles in input:                    ", f"{len(articles):,}")
print("Articles with no symbols at all:      ", f"{int(articles['n_symbols_total'].eq(0).sum()):,}")
print("Articles with >=1 project ticker:     ", f"{int(has_project.sum()):,}")
print("Articles with no project ticker (out):", f"{int((~has_project).sum()):,}")
print("Articles mentioning 2+ project tickers:", f"{int(articles['n_project_tickers'].gt(1).sum()):,}")

## 3. Build `news_tickers.csv` (one row per article–ticker pair)

Only articles with at least one project ticker appear here. Out-of-scope tickers are dropped; the article itself stays in `news_articles_sentiment.csv`.

In [ ]:
mapping = (
    articles.loc[has_project, ["article_id", "project_tickers"]]
    .explode("project_tickers")
    .rename(columns={"project_tickers": "ticker"})
    .drop_duplicates(subset=["article_id", "ticker"])
    .sort_values(["article_id", "ticker"])
    .reset_index(drop=True)
)

assert not mapping.duplicated(["article_id", "ticker"]).any()
assert mapping["ticker"].isin(PROJECT_TICKERS).all()
assert len(mapping) == int(articles.loc[has_project, "n_project_tickers"].sum())

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
mapping.to_csv(TICKERS_PATH, index=False)
print(f"Saved {len(mapping):,} article-ticker associations to {TICKERS_PATH.resolve()}")
display(mapping.head(10))

## 4. Build `news_clean.csv` (flattened: one row per article–ticker pair)

Joins article metadata and sentiment onto each pair. `news_id` is a generated row ID; `article_id` is the original Alpaca ID. `date` is the **UTC calendar date** of publication; the market-reaction date (after close, weekends, holidays) is a separate later derivation.

In [ ]:
article_columns = [
    "article_id", "published_at", "title", "summary", "source", "url",
    "n_project_tickers", "n_symbols_total",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
]

flat = mapping.merge(
    articles[article_columns], on="article_id", how="left", validate="many_to_one"
)
flat["date"] = flat["published_at"].dt.strftime("%Y-%m-%d")

flat = flat.sort_values(["article_id", "ticker"]).reset_index(drop=True)
flat.insert(0, "news_id", range(1, len(flat) + 1))

flat = flat[[
    "news_id", "article_id", "ticker", "published_at", "date", "title", "summary", "source", "url",
    "n_project_tickers", "n_symbols_total",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
]]

# ---- Integrity checks ----
assert len(flat) == len(mapping), "Join changed the number of rows."
assert not flat.duplicated(["article_id", "ticker"]).any()
assert flat["news_id"].is_unique
assert flat["ticker"].isin(PROJECT_TICKERS).all()

flat.to_csv(FLAT_PATH, index=False)
print(f"Saved {len(flat):,} rows to {FLAT_PATH.resolve()}")
display(flat.head(10))

## 5. Quality checks

A project ticker with zero articles, or a year with very few, points to a collection gap upstream (the earlier raw CSV had many empty months), not a mapping bug.

In [ ]:
summary = {
    "unique_articles_in_input": int(len(articles)),
    "articles_with_project_ticker": int(mapping["article_id"].nunique()),
    "articles_dropped_no_project_ticker": int((~has_project).sum()),
    "article_ticker_pairs": int(len(mapping)),
    "flattened_rows": int(len(flat)),
    "missing_sentiment_label_rows": int(flat["finbert_label"].isna().sum()),
    "missing_sentiment_score_rows": int(flat["finbert_score"].isna().sum()),
    "missing_published_at_rows": int(flat["published_at"].isna().sum()),
}
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

per_ticker = (
    mapping["ticker"].value_counts().reindex(TICKERS, fill_value=0)
    .rename_axis("ticker").to_frame("article_count")
)
display(per_ticker)
if per_ticker["article_count"].eq(0).any():
    print("WARNING: tickers with zero articles:", per_ticker.index[per_ticker["article_count"].eq(0)].tolist())

year = flat["published_at"].dt.year
display(pd.crosstab(flat["ticker"], year).reindex(TICKERS, fill_value=0))

## Completion checklist

- [ ] `TICKERS` matches the team's canonical company list.
- [ ] `news_tickers.csv` has one row per unique `(article_id, ticker)`, project tickers only.
- [ ] `news_clean.csv` has the same rows plus article metadata and sentiment.
- [ ] FinBERT was not re-run here.
- [ ] Sentiment is documented as article-level, not company-specific.
- [ ] Every project ticker has articles in every year (otherwise investigate upstream collection).
- [ ] Person 4 has the final column names and keys (`article_id`, `ticker`).